In [23]:
import pandas as pd
import numpy as np
import sklearn

In [ ]:
df = pd.read_csv('georgia_100_noisy.csv')
df.head()
# 0 = low risk
# 1 = high risk 

,city,alert_level,gauge_ratio,risk_label
0,Columbus,0.33,0.000,0
1,Atlanta,0.00,0.064,0
2,Augusta,0.33,0.451,0
3,Savannah,1.00,0.268,1
4,Athens,1.00,1.000,1


In [25]:
# clean nulls
data = df.dropna()
# data = data.drop(["rainfall_norm", "elevation_risk", "risk_category", "scenario"], axis = 1)


In [26]:
# one hot encoding with city names 

data_encode = pd.get_dummies(data, columns=["city"], dtype=int)
data_encode.head()

,alert_level,gauge_ratio,risk_label,city_Albany,city_Athens,city_Atlanta,city_Augusta,city_Columbus,city_Macon,city_Roswell,city_Savannah,city_St. Simons
0,0.33,0.000,0,0,0,0,0,1,0,0,0,0
1,0.00,0.064,0,0,0,1,0,0,0,0,0,0
2,0.33,0.451,0,0,0,0,1,0,0,0,0,0
3,1.00,0.268,1,0,0,0,0,0,0,0,1,0
4,1.00,1.000,1,0,1,0,0,0,0,0,0,0


In [27]:
# divide into target and features
y = data_encode['risk_label']
X = data_encode.drop(['risk_label'], axis = 1)

In [28]:
X.head()

,alert_level,gauge_ratio,city_Albany,city_Athens,city_Atlanta,city_Augusta,city_Columbus,city_Macon,city_Roswell,city_Savannah,city_St. Simons
0,0.33,0.000,0,0,0,0,1,0,0,0,0
1,0.00,0.064,0,0,1,0,0,0,0,0,0
2,0.33,0.451,0,0,0,1,0,0,0,0,0
3,1.00,0.268,0,0,0,0,0,0,0,1,0
4,1.00,1.000,0,1,0,0,0,0,0,0,0


In [29]:
from sklearn.preprocessing import StandardScaler

city_cols = X.iloc[:, 2:] # city cols is df of everything but the two metrics 

scale_cols  = ['alert_level', 'gauge_ratio'] # dont normalize the city numbers 

scaler = StandardScaler()
scaled = scaler.fit_transform(X[scale_cols])

X_np = np.hstack([
    scaled,                          # alert_level, gauge_ratio (scaled)         # rainfall_norm, elevation_risk (untouched)
    city_cols.values             # city dummies (untouched)
])

In [30]:
X_np.shape

(100, 11)

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_np, y, test_size=0.3,  random_state=30, stratify=y)



In [40]:
y.head()

0    0
1    0
2    0
3    1
4    1
Name: risk_label, dtype: int64

In [32]:
X_train.shape

(70, 11)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
lr = LogisticRegression()
lr.fit(X_train, y_train)

pred = lr.predict(X_test)
proba = lr.predict_proba(X_test)[:, 1] # returns prob class 0, prob class 1 


print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))
print("1 = HIGH RISK, 0 = LOW RISK")

print("Training score:", lr.score(X_train, y_train))
print("Test score:    ", lr.score(X_test, y_test))


Accuracy: 0.9666666666666667
ROC AUC: 0.9911111111111112
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        15
           1       1.00      0.93      0.97        15

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

1 = HIGH RISK, 0 = LOW RISK
Training score: 0.9428571428571428
Test score:     0.9666666666666667


In [36]:
# get weights and bias
print("Weights (coef_):", lr.coef_) 
print("Bias (intercept_):", lr.intercept_)

Weights (coef_): [[ 2.45214484  0.63302691  0.87769746 -0.32605279  0.27168987  0.29726719
  -0.45038642 -0.34021768  0.03863536 -0.19321195 -0.17811645]]
Bias (intercept_): [-0.01267626]


In [37]:
import joblib
joblib.dump(lr, "model.pkl")

['model.pkl']